## Sentiment Modeling & Evaluasi

**Model Fallback (Baseline):** TF-IDF + Support Vector Machine (SVM)
**Model Utama (Deep Learning):** Fine-tuning IndoBERT (`indobenchmark/indobert-base-p1`)

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan untuk training: {device}")

### Load Dataset

In [ ]:
dataset_path = '../preprocessing/cleaned_reviews.csv'
if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset bersih berhasil dimuat! Jumlah data: {len(df)} baris.")

    # Bersihkan ulasan kosong
    df = df.dropna(subset=['text_clean', 'sentiment'])
    print(f"Jumlah ulasan setelah dibersihkan dari nilai kosong: {len(df)} baris.")
else:
    print(f"[ERROR] File {dataset_path} tidak ditemukan! Pastikan file berada di folder preprocessing.")

### Dataset Splitting (Stratified Train/Test Split)

In [ ]:
# Membagi dataset menjadi Train (80%) dan Test (20%) secara seimbang (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    df['text_clean'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['sentiment']
)

print(f"Distribusi kelas ulasan pada data latih:\n{y_train.value_counts()}\n")
print(f"Distribusi kelas ulasan pada data uji:\n{y_test.value_counts()}\n")

### Model Fallback: TF-IDF + SVM Classifier

In [ ]:
print("Mengekstrak fitur menggunakan TF-IDF...")
# Ekstraksi fitur kata unigram dan bigram dengan batasan fitur maksimal 5000
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Melatih model baseline SVM dengan bobot kelas seimbang...")
# Gunakan linear kernel yang stabil untuk text classification
svm_model = SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42)
svm_model.fit(X_train_tfidf, y_train)
print("Model SVM selesai dilatih!")

In [ ]:
# Prediksi & Evaluasi Model Baseline SVM
svm_preds = svm_model.predict(X_test_tfidf)

print("--- HASIL EVALUASI BASELINE SVM ---")
print(classification_report(y_test, svm_preds))

### Model Utama: Fine-Tuning IndoBERT
#### Tokenisasi Dataset

In [ ]:
# tokenizer untuk IndoBERT-base
model_name = 'indobenchmark/indobert-base-p1'
print(f"Mengunduh Tokenizer untuk {model_name}...")
tokenizer = BertTokenizer.from_pretrained(model_name)

# Mapping sentimen string ke nilai numerik integer
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
y_train_mapped = y_train.map(label_map).values
y_test_mapped = y_test.map(label_map).values

def tokenize_data(texts, max_len=128):
    input_ids = []
    attention_masks = []
    
    for text in texts:
        encoded = tokenizer.encode_plus(
            str(text),
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
        
    return torch.cat(input_ids, dim=0), torch.cat(attention_masks, dim=0)

print("Melakukan tokenisasi data latih...")
train_inputs, train_masks = tokenize_data(X_train)
print("Melakukan tokenisasi data uji...")
test_inputs, test_masks = tokenize_data(X_test)

train_labels = torch.tensor(y_train_mapped)
test_labels = torch.tensor(y_test_mapped)

print("Tokenisasi Selesai!")

#### PyTorch DataLoaders

In [ ]:
# Membuat PyTorch Dataset dan DataLoaders
batch_size = 16

train_dataset = TensorDataset(train_inputs, train_masks, train_labels)
train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)

test_dataset = TensorDataset(test_inputs, test_masks, test_labels)
test_loader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=batch_size)

print(f"DataLoader siap dengan Batch Size: {batch_size}")

#### Training Loop (Fine-Tuning)

In [ ]:
# Load model klasifikasi IndoBERT dengan 3 output kelas (negative, neutral, positive)
print(f"Mengunduh dan menginisialisasi model {model_name}...")
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)
model.to(device)

# Gunakan AdamW optimizer yang direkomendasikan untuk Transformer
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# Set Epoch pelatihan
epochs = 3
print(f"Memulai proses fine-tuning IndoBERT sebanyak {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for step, batch in enumerate(train_loader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        model.zero_grad()
        
        outputs = model(
            b_input_ids, 
            token_type_ids=None, 
            attention_mask=b_input_mask, 
            labels=b_labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        if step % 20 == 0 and step > 0:
            print(f"Epoch: {epoch + 1} | Batch: {step}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_loss / len(train_loader)
    print(f"[RESULT] Epoch {epoch + 1}/{epochs} | Average Training Loss: {avg_train_loss:.4f}\n")

### Evaluasi IndoBERT & Visualisasi Performa

In [ ]:
# Set model ke mode evaluasi
model.eval()
bert_preds = []

print("Melakukan prediksi pada data uji menggunakan IndoBERT...")
with torch.no_grad():
    for batch in test_loader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        logits = outputs.logits.detach().cpu().numpy()
        # Ambil indeks kelas dengan probabilitas tertinggi
        bert_preds.extend(np.argmax(logits, axis=1))

# Mengembalikan label numerik ke label asalnya (string)
reverse_label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
bert_preds_decoded = [reverse_label_map[p] for p in bert_preds]

print("--- HASIL EVALUASI INDOBERT ---")
print(classification_report(y_test, bert_preds_decoded))

In [ ]:
# Visualisasi Perbandingan Model (Akurasi)
svm_acc = accuracy_score(y_test, svm_preds)
bert_acc = accuracy_score(y_test, bert_preds_decoded)

models = ['SVM (TF-IDF)', 'IndoBERT (Fine-Tuning)']
accuracies = [svm_acc * 100, bert_acc * 100]

plt.figure(figsize=(6, 4))
sns.barplot(x=models, y=accuracies, palette='Set2')
plt.ylabel('Akurasi (%)')
plt.title('Perbandingan Akurasi Model')
for i, v in enumerate(accuracies):
    plt.text(i, v + 1, f"{v:.2f}%", ha='center', fontweight='bold')
plt.ylim(0, 100)
plt.show()

In [ ]:
# Plot Confusion Matrix untuk IndoBERT
cm = confusion_matrix(y_test, bert_preds_decoded, labels=['negative', 'neutral', 'positive'])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Negative', 'Neutral', 'Positive'], 
    yticklabels=['Negative', 'Neutral', 'Positive']
)
plt.ylabel('Ground Truth')
plt.xlabel('Prediksi')
plt.title('Confusion Matrix - IndoBERT')
plt.show()